## 1. Petitions Data Ingestion
The data for this project comes from 2 sources: csv files downloaded from the petitions website (here: https://petition.parliament.uk/petitions?state=all and for archives: https://petition.parliament.uk/archived/petitions?parliament=4&state=all) and JSON files. Files are currently split by parliaments but will be merged to create metadata file and master file for analysis.\
\
The url from the metadata csv file is used as an index to iterate through the API and access each petitions json file. For this process, two functions are continually referenced for each petition. Each petition has text data about the petition and lists of signatures by country, region and constituency which are each broken down into their own file. Hence, there are 4 files created from the API: petitions_text_data, petitions_countries_data, petitions_regions_data and petitions_constituency_data. To increase processing speed and for ease of access after the data has first been ingested from the API, the resulting files are saved to parquet files.\
\
This project has made use of literature and similar projects on Github etc., which are referenced at the end of each section.\
\
Due to the large number of petitions (116,000), the corpus cannot be manually checked and read and so accuracy will be tested on a much smaller sample. I use an unsupervised NLP algorithm, latent Dirichlet allocation (LDA) to extract topics from all of the free-text columns.\
\
For testing, the sample data will be split into partitions and a model is trained on some of the partitions then tested on the last partition. I will then repeat this process. If several rounds of testing give similar results, the model will be shown as stable.

In [ ]:
# importing all relevant packages
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
# extracting the data from csv files and adding '.json' to url to match up with petitions extracted from json
metadata_2025 = pd.read_csv('data/all-petitions-2025.csv')
metadata_2025['url'] = metadata_2025['URL'] + '.json'
metadata_2019_2024 = pd.read_csv('data/all-petitions-2019-2024.csv')
metadata_2019_2024['url'] = metadata_2019_2024['URL'] + '.json'
metadata_2017_2019 = pd.read_csv('data/all-petitions-2017-2019.csv')
metadata_2017_2019['url'] = metadata_2017_2019['URL'] + '.json'
metadata_2015_2017 = pd.read_csv('data/all-petitions-2015-2017.csv')
metadata_2015_2017['url'] = metadata_2015_2017['URL'] + '.json'

In [3]:
# merging the metadata files for later reference
petitions_meta_df = pd.concat([metadata_2025, metadata_2019_2024, metadata_2017_2019, metadata_2015_2017])

petitions_meta_df.head(10)

,Petition,URL,State,Signatures Count,url
0,Fund reconstruction surgery and psychosexual t...,https://petition.parliament.uk/petitions/700001,closed,939,https://petition.parliament.uk/petitions/70000...
1,Apply for the UK to join the European Union as...,https://petition.parliament.uk/petitions/700005,closed,136651,https://petition.parliament.uk/petitions/70000...
2,Rejoin the European Union,https://petition.parliament.uk/petitions/700006,rejected,10,https://petition.parliament.uk/petitions/70000...
3,Require dating apps to ensure ID and DBS check...,https://petition.parliament.uk/petitions/700007,closed,1216,https://petition.parliament.uk/petitions/70000...
4,"Fund Covid booster vaccines for all, with prio...",https://petition.parliament.uk/petitions/700009,closed,716,https://petition.parliament.uk/petitions/70000...
5,Only allow alcohol to be sold in pubs and in w...,https://petition.parliament.uk/petitions/700010,closed,90,https://petition.parliament.uk/petitions/70001...
6,Donate migrant boats to Ukraine,https://petition.parliament.uk/petitions/700011,closed,436,https://petition.parliament.uk/petitions/70001...
7,Grant undocumented migrants the right to live ...,https://petition.parliament.uk/petitions/700012,rejected,12,https://petition.parliament.uk/petitions/70001...
8,Limit the sale and use of Fireworks to licence...,https://petition.parliament.uk/petitions/700013,closed,87528,https://petition.parliament.uk/petitions/70001...
9,Decriminalise Abortion,https://petition.parliament.uk/petitions/700014,closed,103653,https://petition.parliament.uk/petitions/70001...


In [4]:
# checking shape as a guide for how many petitions to expect to come from the API
petitions_meta_df.shape

(116979, 5)

In [5]:
# saving to csv for future use
petitions_meta_df.to_csv('data\petitions_metadata.csv')
# for future use, will be stored under new variable
petitions_metadata_df = pd.read_csv('data\petitions_metadata.csv')

<>:2: SyntaxWarning: invalid escape sequence '\p'
<>:4: SyntaxWarning: invalid escape sequence '\p'
<>:2: SyntaxWarning: invalid escape sequence '\p'
<>:4: SyntaxWarning: invalid escape sequence '\p'
C:\Users\morriseb\AppData\Local\Temp\ipykernel_21876\948665642.py:2: SyntaxWarning: invalid escape sequence '\p'
  petitions_meta_df.to_csv('data\petitions_metadata.csv')
C:\Users\morriseb\AppData\Local\Temp\ipykernel_21876\948665642.py:4: SyntaxWarning: invalid escape sequence '\p'
  petitions_metadata_df = pd.read_csv('data\petitions_metadata.csv')


Once the metadata has been ingested, the API needs to be queried to get more text data for each petition. To do this, the url column from the metadata csv is used as an index to ensure every petition is accounted for in the process. There are 4 files that will be created as a result: 'petitions_text_data', 'petitions_region_data', 'petitions_countries_data' and 'petitions_constituency_data'. These files will be kept seperate as combining could lead to confusion and miscounting signatures due to change in granularity. 

In [6]:
def get_text_data(url, parliament):
    '''
    takes the petition url and queries the API for a response. If there is a response then returns petition data and if not just returns petition ID and url
    :param url: - string, provides parameter for query
    :output: 4 dictionaries corresponding to petition text data, signatures by country, region and constituency
    '''
    while True:
        # send a request to the server
        response = requests.get(url)
        # saving response to a variable and extracting list of items from the response
        try:
            petition_resp = response.json()
            petition = petition_resp.get("data", []) #access list of petitions under "data" key
        except:
            petition = False
        if not petition:
            # some petitions are being checked to see if meet requirements and so no response comes back from the url. In this case, basic info on ID and url is given and the rest
            # can be blank. This allows all petitions to be counted but will make a note that all petitions with blank "action" can be considered as having no response and will
            # not be considered in the analysis.
            petition_data = {
                "petition_id": int(url[-11:-5]),
                "url": url,
                "action": "",
                "background": "",
                "details": "",
                "created_at": "",
                "signature_count": 0,
                "parliament": parliament

            }
            country_data = {}
            region_data = {}
            constituency_data = {}
            break
        else:
            # saving relevant values to dictionary to be turned into dataframe
            petition_data = {
                "petition_id": petition.get("id"),
                "url": url,
                "action": petition.get("attributes", {}).get("action"),
                "background": petition.get("attributes", {}).get("background"),
                "details": petition.get("attributes", {}).get("additional_details"),
                "created_at": petition.get("attributes", {}).get("created_at"),
                "signature_count": int(petition.get("attributes", {}).get("signature_count")),
                "parliament": parliament

            }
            # storing the location and signature data as a dictionary and then looping through to add the petition id as a seperate key
            country_data = petition.get("attributes", []).get("signatures_by_country", [])
            for country in country_data:
                country['petition_id'] = petition.get("id")
            region_data = petition.get("attributes", []).get("signatures_by_region", [])
            for region in region_data:
                region['petition_id'] = petition.get("id")
            constituency_data = petition.get("attributes", []).get("signatures_by_constituency", [])
            for constituency in constituency_data:
                constituency['petition_id'] = petition.get("id")
            break
    return petition_data, country_data, region_data, constituency_data

In [7]:
def extract_files(files, parliament):
    '''
    takes the metadata file for a given parliament and uses it as an index to obtain data for each petition with the get_text_data function
    :param files: - dataframe, index of petitions
    :output: 4 dataframes containing data on all petitions from given parliament
    '''
    # setting up empty lists and dataframes which will be incrementally appended to
    petitions_info = []
    petition_by_country = pd.DataFrame()
    petitions_by_region = pd.DataFrame()
    petition_by_constituency = pd.DataFrame()

    # looping through the csv file to get data for each petition
    for url in files['url']:
        #getting the data for each petition
        petition_data, country_data, region_data, constituency_data = get_text_data(url, parliament)
        # adding the data onto each dataframe/list to create a merged table
        petitions_info.append(petition_data)
        country_df = pd.DataFrame(country_data)
        petition_by_country = pd.concat([petition_by_country, country_df])
        region_df = pd.DataFrame(region_data)
        petitions_by_region = pd.concat([petitions_by_region, region_df])
        constituency_df = pd.DataFrame(constituency_data)
        petition_by_constituency = pd.concat([petition_by_constituency, constituency_df])
    petitions_text_data = pd.DataFrame(petitions_info)
    return petitions_text_data, petition_by_country, petitions_by_region, petition_by_constituency

Running through a smaller sample first (just using petitions data for 2025). First extracting the data and then having a look at the format and first few values of the data to check everything is as expected.

In [8]:
# calling the functions seperately for each parlaiment (as it will take a long time to run)
petitions_text_data_2025, petition_countries_2025, petition_regions_2025, petition_constituency_2025 = extract_files(metadata_2025, 4)
# adding primary key
petition_countries_2025["ID"] = petition_countries_2025.index
petition_regions_2025["ID"] = petition_regions_2025.index
petition_constituency_2025["ID"] = petition_constituency_2025.index

In [9]:
petition_countries_2025.head()

,name,code,signature_count,petition_id,ID
0,Australia,AU,1,700001,0
1,Canada,CA,1,700001,1
2,Cyprus,CY,1,700001,2
3,Dominican Republic,DO,1,700001,3
4,Germany,DE,1,700001,4


In [10]:
petition_regions_2025.head()

,name,ons_code,signature_count,petition_id,ID
0,North East,A,32,700001,0
1,North West,B,105,700001,1
2,Yorkshire and The Humber,D,62,700001,2
3,East Midlands,E,55,700001,3
4,West Midlands,F,64,700001,4


In [11]:
petition_constituency_2025.head()

,name,ons_code,mp,signature_count,petition_id,ID
0,Aldershot,E14001063,Alex Baker MP,3,700001,0
1,Altrincham and Sale West,E14001065,Mr Connor Rand MP,2,700001,1
2,Amber Valley,E14001066,Linsey Farnsworth MP,2,700001,2
3,Ashfield,E14001068,Lee Anderson MP,2,700001,3
4,Ashton-under-Lyne,E14001070,Rt Hon Angela Rayner MP,2,700001,4


In [12]:
petitions_text_data_2025.head()

,petition_id,url,action,background,details,created_at,signature_count,parliament
0,700001,https://petition.parliament.uk/petitions/70000...,Fund reconstruction surgery and psychosexual t...,We urge the Government to support FGM survivor...,I know firsthand the physical and emotional ch...,2024-10-30T14:20:17.653Z,939,4
1,700005,https://petition.parliament.uk/petitions/70000...,Apply for the UK to join the European Union as...,I believe joining the EU would boost the econo...,I think the Single Market would bring inward i...,2024-10-30T14:28:44.914Z,136651,4
2,700006,https://petition.parliament.uk/petitions/70000...,Rejoin the European Union,The UK Government should seek to rejoin the Eu...,The real black hole in the UK’s economy is all...,2024-10-30T14:32:33.904Z,10,4
3,700007,https://petition.parliament.uk/petitions/70000...,Require dating apps to ensure ID and DBS check...,I believe to better protect the public from of...,I believe that the ability for violent offende...,2024-10-30T14:34:44.936Z,1216,4
4,700009,https://petition.parliament.uk/petitions/70000...,"Fund Covid booster vaccines for all, with prio...",Long Covid can ruin lives and cause financial ...,"Currently, many Long Covid sufferers are unabl...",2024-10-30T14:53:44.200Z,716,4


Now ingesting on a larger scale as the dataframes are returning data as expected and saving to parquet files

In [14]:
petitions_text_data_2019, petition_countries_2019, petition_regions_2019, petition_constituency_2019 = extract_files(metadata_2019_2024, 3)

In [15]:
petitions_text_data_2017, petition_countries_2017, petition_regions_2017, petition_constituency_2017 = extract_files(metadata_2017_2019, 2)

In [16]:
petitions_text_data_2015, petition_countries_2015, petition_regions_2015, petition_constituency_2015 = extract_files(metadata_2015_2017, 1)

In [17]:
petitions_text_data = pd.concat([petitions_text_data_2025, petitions_text_data_2019, petitions_text_data_2017, petitions_text_data_2015])
petitions_countries = pd.concat([petition_countries_2025, petition_countries_2019, petition_countries_2017, petition_countries_2015])
petitions_region = pd.concat([petition_regions_2025, petition_regions_2019, petition_regions_2017, petition_regions_2015])
petitions_constituency = pd.concat([petition_constituency_2025, petition_constituency_2019, petition_constituency_2017, petition_constituency_2015]) 

petitions_countries["ID"] = petitions_countries.index
petitions_region["ID"] = petitions_region.index
petitions_constituency["ID"] = petitions_constituency.index

In [18]:
petitions_text_pa = pa.Table.from_pandas(petitions_text_data)
petitions_countries_pa = pa.Table.from_pandas(petitions_countries)
petitions_region_pa = pa.Table.from_pandas(petitions_region)
petitions_constituency_pa = pa.Table.from_pandas(petitions_constituency)

pq.write_table(petitions_text_pa, 'data\petitions_data.parquet')
pq.write_table(petitions_countries_pa, 'data\petitions_country_data.parquet')
pq.write_table(petitions_region_pa, 'data\petitions_region_data.parquet')
pq.write_table(petitions_constituency_pa, 'data\petitions_constituency_data.parquet')

<>:6: SyntaxWarning: invalid escape sequence '\p'
<>:7: SyntaxWarning: invalid escape sequence '\p'
<>:8: SyntaxWarning: invalid escape sequence '\p'
<>:9: SyntaxWarning: invalid escape sequence '\p'
<>:6: SyntaxWarning: invalid escape sequence '\p'
<>:7: SyntaxWarning: invalid escape sequence '\p'
<>:8: SyntaxWarning: invalid escape sequence '\p'
<>:9: SyntaxWarning: invalid escape sequence '\p'
C:\Users\morriseb\AppData\Local\Temp\ipykernel_21876\2001151391.py:6: SyntaxWarning: invalid escape sequence '\p'
  pq.write_table(petitions_text_pa, 'data\petitions_data.parquet')
C:\Users\morriseb\AppData\Local\Temp\ipykernel_21876\2001151391.py:7: SyntaxWarning: invalid escape sequence '\p'
  pq.write_table(petitions_countries_pa, 'data\petitions_country_data.parquet')
C:\Users\morriseb\AppData\Local\Temp\ipykernel_21876\2001151391.py:8: SyntaxWarning: invalid escape sequence '\p'
  pq.write_table(petitions_region_pa, 'data\petitions_region_data.parquet')
C:\Users\morriseb\AppData\Local\Tem